# Random forest analysis estimating variance in Fitbit sleep explained by different feature sets

In [3]:
import warnings
from arivale_data_interface import get_snapshot
import pandas as pd
import numpy as np
import scipy
import json

warnings.simplefilter("ignore")

#import the final master sleep_merged df and the features_dict

sleep_merged = pd.read_csv('../working_df/sleep_merged_sleep_meds_excluded_10_14_2025.csv', dtype={'public_client_id': object})

with open('../working_df/features_dict.json', 'r') as f:
    features_dict = json.load(f)

In [4]:
# Organize sleep variables into higher level groups:
sleep_duration_variables = ['sleep_minutesAsleep_log1p_resid',
                            'sleep_timeInBed_log1p_resid']
sleep_disruption_variables = ['sleep_awakeningsCount_log1p_resid',
                              'sleep_minutesAwake_log1p_resid',
                              'sleep_awakeDuration_log1p_resid',
                              'sleep_restlessDuration_log1p_resid',
                              'sleep_minutesAfterWakeup_log1p_resid']
sleep_duration_variability_variables = ['sleep_minutesAsleep_std_log1p_resid',
                                        'sleep_timeInBed_std_log1p_resid']
sleep_disruption_variability_variables = ['sleep_awakeningsCount_std_log1p_resid',
                                          'sleep_minutesAwake_std_log1p_resid',
                                          'sleep_awakeDuration_std_log1p_resid',
                                          'sleep_restlessDuration_std_log1p_resid',
                                          'sleep_minutesAfterWakeup_std_log1p_resid']

In [5]:
features_dict.keys()

dict_keys(['heartrate_features', 'activity_features', 'sleep_features', 'covariate_features', 'sleep_features_log1p', 'sleep_features_log1p_resids', 'activity_features_resids', 'heartrate_features_resids', 'vendor_features', 'new_microbe_binarized_features', 'new_microbe_features_binary_10_90', 'new_microbe_features', 'diversity_features', 'dip_features', 'metabolite_features', 'food_freq_features', 'food_freq_features_daily', 'food_freq_from_non_freq_features', 'food_freq_from_non_freq_features_daily', 'diet_daily_features', 'gut_q_features', 'mental_health_q_features', 'sleep_disorder_q_features', 'digestion_features', 'supplement_features', 'med_features', 'clinical_features', 'proteomics_features', 'micom_features', 'metabolite_features_log', 'clinical_features_log1p', 'micom_features_log', 'happiness_features', 'happiness_features_numeric'])

In [6]:
features_dict['vendor_features']

['microbiome_vendor',
 'diet_vendor',
 'mental_health_gut_sleep_q_vendor',
 'digestion_vendor',
 'supp_meds_vendor',
 'clinical_vendor',
 'happiness_vendor']

In [7]:
# No need to include digestion vendor or supp_meds_vendor bc it is identical to mental health gut sleep q vendor
vendor_features_for_rf = ['supp_meds_vendor', 'clinical_vendor', 'mental_health_gut_sleep_q_vendor', 'microbiome_vendor']

In [8]:
set(sleep_merged['digestion_vendor'].dropna())==(set(sleep_merged['mental_health_gut_sleep_q_vendor'].dropna()))

True

In [9]:
set(sleep_merged['supp_meds_vendor'].dropna())==(set(sleep_merged['mental_health_gut_sleep_q_vendor'].dropna()))

True

In [10]:
sleep_merged[vendor_features_for_rf].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1469 entries, 0 to 1468
Data columns (total 4 columns):
 #   Column                            Non-Null Count  Dtype 
---  ------                            --------------  ----- 
 0   supp_meds_vendor                  1469 non-null   object
 1   clinical_vendor                   1148 non-null   object
 2   mental_health_gut_sleep_q_vendor  1469 non-null   object
 3   microbiome_vendor                 1469 non-null   object
dtypes: object(4)
memory usage: 46.0+ KB


In [11]:
covariate_features = ['age', 'BMI_CALC', 'PC1', 'PC2', 'PC3', 'sex',
                      'any_mental_health_self_anytime_before', 'any_sleep_disorder_self_anytime_before',
                      'avg_stress']

In [10]:
from statsmodels.formula.api import ols
    
sleep_merged_rf = sleep_merged.copy()
    
for vendor_col in vendor_features_for_rf:
    sleep_merged_rf[vendor_col] = sleep_merged_rf[vendor_col].fillna('missing')
    
# Generate wrapped vendor variables
vendor_terms = " + ".join([f'C({v})' for v in vendor_features_for_rf])
    
for feature in features_dict['sleep_features_log1p_resids']:
    dropped_df = sleep_merged_rf.dropna(
        subset=[feature] + vendor_features_for_rf
    )
    
    formula = f"{feature} ~ {vendor_terms}"
    
    fitted = ols(formula, data=dropped_df).fit()
    
    # Store residuals
    sleep_merged_rf.loc[dropped_df.index, f'{feature}_rf_vendor_only_resid'] = fitted.resid
    print(formula)
    
# List of vendor ONLY residualized sleep features
sleep_features_vendor_only_resid_rf = [f'{feature}_rf_vendor_only_resid' for feature in features_dict['sleep_features_log1p_resids']]

sleep_awakeDuration_log1p_resid ~ C(supp_meds_vendor) + C(clinical_vendor) + C(mental_health_gut_sleep_q_vendor) + C(microbiome_vendor)
sleep_awakeningsCount_log1p_resid ~ C(supp_meds_vendor) + C(clinical_vendor) + C(mental_health_gut_sleep_q_vendor) + C(microbiome_vendor)
sleep_efficiency_log1p_resid ~ C(supp_meds_vendor) + C(clinical_vendor) + C(mental_health_gut_sleep_q_vendor) + C(microbiome_vendor)
sleep_minutesAfterWakeup_log1p_resid ~ C(supp_meds_vendor) + C(clinical_vendor) + C(mental_health_gut_sleep_q_vendor) + C(microbiome_vendor)
sleep_minutesAsleep_log1p_resid ~ C(supp_meds_vendor) + C(clinical_vendor) + C(mental_health_gut_sleep_q_vendor) + C(microbiome_vendor)
sleep_minutesAwake_log1p_resid ~ C(supp_meds_vendor) + C(clinical_vendor) + C(mental_health_gut_sleep_q_vendor) + C(microbiome_vendor)
sleep_restlessDuration_log1p_resid ~ C(supp_meds_vendor) + C(clinical_vendor) + C(mental_health_gut_sleep_q_vendor) + C(microbiome_vendor)
sleep_timeInBed_log1p_resid ~ C(supp_meds_

In [11]:
sleep_merged_rf[sleep_features_vendor_only_resid_rf].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1469 entries, 0 to 1468
Data columns (total 18 columns):
 #   Column                                                         Non-Null Count  Dtype  
---  ------                                                         --------------  -----  
 0   sleep_awakeDuration_log1p_resid_rf_vendor_only_resid           898 non-null    float64
 1   sleep_awakeningsCount_log1p_resid_rf_vendor_only_resid         1469 non-null   float64
 2   sleep_efficiency_log1p_resid_rf_vendor_only_resid              1469 non-null   float64
 3   sleep_minutesAfterWakeup_log1p_resid_rf_vendor_only_resid      1469 non-null   float64
 4   sleep_minutesAsleep_log1p_resid_rf_vendor_only_resid           898 non-null    float64
 5   sleep_minutesAwake_log1p_resid_rf_vendor_only_resid            497 non-null    float64
 6   sleep_restlessDuration_log1p_resid_rf_vendor_only_resid        898 non-null    float64
 7   sleep_timeInBed_log1p_resid_rf_vendor_only_resid            

In [12]:
sleep_merged_rf[features_dict['digestion_features']].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1469 entries, 0 to 1468
Data columns (total 16 columns):
 #   Column                               Non-Null Count  Dtype 
---  ------                               --------------  ----- 
 0   digestion_abdominal_pain             1072 non-null   object
 1   digestion_acid_reflux                1293 non-null   object
 2   digestion_bloating                   1293 non-null   object
 3   digestion_bowel_movement_completion  1072 non-null   object
 4   digestion_bowel_movement_ease        1072 non-null   object
 5   digestion_bowel_movements            1072 non-null   object
 6   digestion_diarrhea                   1293 non-null   object
 7   digestion_gas                        1072 non-null   object
 8   digestion_high_appetite              1072 non-null   object
 9   digestion_laxatives                  1072 non-null   object
 10  digestion_medications                1072 non-null   object
 11  digestion_nausea                     1293 n

In [13]:
sleep_merged_rf[features_dict['digestion_features']]

,digestion_abdominal_pain,digestion_acid_reflux,digestion_bloating,digestion_bowel_movement_completion,digestion_bowel_movement_ease,digestion_bowel_movements,digestion_diarrhea,digestion_gas,digestion_high_appetite,digestion_laxatives,digestion_medications,digestion_nausea,digestion_poor_or_lack_of_appetite,digestion_stress_eating,digestion_supplements,digestion_vomiting
0,(1) Not at all,(1) Not at all,(1) Infrequently or not at all,(3) Most of the time,(2) Sometimes difficult,(2) 3-6 times per week,(1) Infrequently or not at all,(1) Infrequently or not at all,(2) Occasionally (once a week or less),(1) Not at all,(1) Not at all,(1) Infrequently or not at all (less than mont...,(1) No,(3) Several times per week,(1) Not at all,(1) Occasionally (once a week or less)
1,(1) Not at all,(1) Not at all,(1) Infrequently or not at all,(3) Most of the time,(1) Easy to pass,(3) 1-3 times daily,(2) Once per week or less,(4) Daily,(2) Occasionally (once a week or less),(1) Not at all,(1) Not at all,(1) Infrequently or not at all (less than mont...,(1) No,(2) Once per week or less,(2) Once per week or less,(1) Occasionally (once a week or less)
2,(1) Not at all,(2) Less than weekly,(1) Infrequently or not at all,(3) Most of the time,(1) Easy to pass,(3) 1-3 times daily,(2) Once per week or less,(3) Several times per week,(2) Occasionally (once a week or less),(1) Not at all,(4) Daily,(1) Infrequently or not at all (less than mont...,(1) No,(3) Several times per week,(1) Not at all,(1) Occasionally (once a week or less)
3,(1) Not at all,(1) Not at all,(1) Infrequently or not at all,(3) Most of the time,(1) Easy to pass,(3) 1-3 times daily,(1) Infrequently or not at all,(1) Infrequently or not at all,(2) Occasionally (once a week or less),(1) Not at all,(1) Not at all,(1) Infrequently or not at all (less than mont...,(1) No,(1) Not at all,(1) Not at all,(1) Occasionally (once a week or less)
4,(1) Not at all,(1) Not at all,(1) Infrequently or not at all,(3) Most of the time,(1) Easy to pass,(3) 1-3 times daily,(1) Infrequently or not at all,(3) Several times per week,(2) Occasionally (once a week or less),(1) Not at all,(1) Not at all,(1) Infrequently or not at all (less than mont...,(1) No,(2) Once per week or less,(4) Daily,(1) Occasionally (once a week or less)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1464,(1) Not at all,(2) Less than weekly,(1) Infrequently or not at all,(3) Most of the time,(1) Easy to pass,(3) 1-3 times daily,(1) Infrequently or not at all,(1) Infrequently or not at all,(2) Occasionally (once a week or less),(1) Not at all,(2) Once per week or less,(1) Infrequently or not at all (less than mont...,(1) No,(1) Not at all,(1) Not at all,(1) Occasionally (once a week or less)
1465,(1) Not at all,(2) Less than weekly,(1) Infrequently or not at all,(3) Most of the time,(1) Easy to pass,(3) 1-3 times daily,(2) Once per week or less,(2) Once per week or less,(3) Regularly (daily or several times per week),(1) Not at all,(2) Once per week or less,(1) Infrequently or not at all (less than mont...,(1) No,(3) Several times per week,(2) Once per week or less,(1) Occasionally (once a week or less)
1466,(1) Not at all,(2) Less than weekly,(1) Infrequently or not at all,(3) Most of the time,(1) Easy to pass,(2) 3-6 times per week,(2) Once per week or less,(1) Infrequently or not at all,(3) Regularly (daily or several times per week),(1) Not at all,(1) Not at all,(1) Infrequently or not at all (less than mont...,(1) No,(2) Once per week or less,(3) Several times per week,(1) Occasionally (once a week or less)
1467,(3) More than 1 time per week,(1) Not at all,(3) Several times per week,(1) Infrequently,(2) Sometimes difficult,(4) 4+ times daily,(2) Once per week or less,(2) Once per week or less,(2) Occasionally (once a week or less),(2) Once per week or less,(1) Not at all,(1) Infrequently or not at all (less than mont...,(1) No,(3) Several times per week,(4) Daily,(1) Occasionally (once a week or less)


In [14]:
# Convert the digestion features to integers for RF
for col in features_dict['digestion_features']:
    sleep_merged_rf[col] = sleep_merged_rf[col].astype('category').cat.codes.replace(-1, np.nan) # <-- by default fills nan with -1, put nan back

In [15]:
sleep_merged_rf[features_dict['digestion_features']].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1469 entries, 0 to 1468
Data columns (total 16 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   digestion_abdominal_pain             1072 non-null   float64
 1   digestion_acid_reflux                1293 non-null   float64
 2   digestion_bloating                   1293 non-null   float64
 3   digestion_bowel_movement_completion  1072 non-null   float64
 4   digestion_bowel_movement_ease        1072 non-null   float64
 5   digestion_bowel_movements            1072 non-null   float64
 6   digestion_diarrhea                   1293 non-null   float64
 7   digestion_gas                        1072 non-null   float64
 8   digestion_high_appetite              1072 non-null   float64
 9   digestion_laxatives                  1072 non-null   float64
 10  digestion_medications                1072 non-null   float64
 11  digestion_nausea              

In [16]:
sleep_merged_rf[features_dict['digestion_features']]

,digestion_abdominal_pain,digestion_acid_reflux,digestion_bloating,digestion_bowel_movement_completion,digestion_bowel_movement_ease,digestion_bowel_movements,digestion_diarrhea,digestion_gas,digestion_high_appetite,digestion_laxatives,digestion_medications,digestion_nausea,digestion_poor_or_lack_of_appetite,digestion_stress_eating,digestion_supplements,digestion_vomiting
0,0.0,0.0,0.0,2.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0
1,0.0,0.0,0.0,2.0,0.0,2.0,1.0,3.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
2,0.0,1.0,0.0,2.0,0.0,2.0,1.0,2.0,1.0,0.0,3.0,0.0,0.0,2.0,0.0,0.0
3,0.0,0.0,0.0,2.0,0.0,2.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,2.0,0.0,2.0,0.0,2.0,1.0,0.0,0.0,0.0,0.0,1.0,3.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1464,0.0,1.0,0.0,2.0,0.0,2.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
1465,0.0,1.0,0.0,2.0,0.0,2.0,1.0,1.0,2.0,0.0,1.0,0.0,0.0,2.0,1.0,0.0
1466,0.0,1.0,0.0,2.0,0.0,1.0,1.0,0.0,2.0,0.0,0.0,0.0,0.0,1.0,2.0,0.0
1467,2.0,0.0,2.0,0.0,1.0,3.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,2.0,3.0,0.0


In [17]:
sleep_merged_rf.sex.value_counts()

sex
F    904
M    565
Name: count, dtype: int64

In [18]:
# Convert sleep_merged_rf sex column to numeric
sleep_merged_rf['sex'] = sleep_merged_rf.sex.replace({'M': 0, 'F': 1})

In [19]:
sleep_merged_rf.sex.value_counts()

sex
1    904
0    565
Name: count, dtype: int64

In [20]:
sleep_merged_rf[features_dict['clinical_features_log1p']].describe()

,a_g_ratio_log1p,adiponectin_serum_log1p,alat_sgpt_log1p,albumin_log1p,alkaline_phosphatase_log1p,antioxid_cap_total_log1p,arachidonic_acid_log1p,arsenic_blood_log1p,asat_sgot_log1p,basophils_log1p,...,vitamin_d3_25_oh_log1p,vldl_ldl_particle_number_log1p,vldl_particle_number_log1p,vldl_size_percentile_log1p,vldl_triglycerides_log1p,white_cell_count_log1p,zinc_log1p,zinc_rbc_log1p,leptin_log1p,zinc_plasma_or_serum_log1p
count,1147.000000,1103.000000,1147.000000,1147.000000,1147.000000,196.000000,1132.000000,196.000000,1147.000000,1127.000000,...,195.000000,31.000000,31.000000,31.000000,31.000000,1129.000000,966.000000,196.000000,26.000000,163.000000
mean,1.038464,2.131424,3.021567,1.684607,4.134677,7.143688,2.409374,1.246140,3.045043,0.383710,...,3.554577,7.099793,3.903722,3.315462,4.111088,1.849999,7.165792,2.438389,3.008439,4.432951
std,0.096765,0.551726,0.454119,0.050551,0.293190,0.306595,0.196115,0.288472,0.288126,0.331390,...,0.327996,0.327077,0.496845,0.985630,0.462610,0.206837,0.184839,0.128968,0.648007,0.131234
min,0.741937,0.470004,1.791759,1.481605,3.044522,6.358189,0.262364,1.098612,2.079442,0.000000,...,2.639057,6.366470,2.844909,0.336472,3.202746,1.193922,6.431331,1.974081,2.128232,4.094345
25%,0.955511,1.740466,2.708050,1.648659,3.951244,6.923727,2.312535,1.098612,2.833213,0.000000,...,3.367296,6.822735,3.593454,2.720884,3.836546,1.722767,7.044905,2.370244,2.455062,4.343805
50%,1.029619,2.128232,2.995732,1.686399,4.143135,7.106729,2.433613,1.098612,3.044522,0.470004,...,3.526361,7.179308,3.992681,3.671225,4.104295,1.840550,7.180831,2.451005,2.871966,4.430817
75%,1.098612,2.517696,3.258097,1.722767,4.330733,7.343794,2.541602,1.386294,3.218876,0.693147,...,3.761200,7.339490,4.247056,3.999792,4.422504,1.987874,7.291656,2.517696,3.458918,4.521789
max,1.360977,3.943522,5.497168,1.840550,5.590987,8.113576,2.815409,2.564949,4.624973,1.098612,...,4.465908,7.583756,4.779123,4.516339,5.126936,2.631889,7.732808,2.772589,4.483003,4.779123


In [21]:
features_dict['activity_features_resids']

['activities_activityCalories_resid',
 'activities_calories_resid',
 'activities_distance_resid',
 'activities_elevation_resid',
 'activities_floors_resid',
 'activities_minutesFairlyActive_resid',
 'activities_minutesLightlyActive_resid',
 'activities_minutesSedentary_resid',
 'activities_minutesVeryActive_resid',
 'activities_steps_resid',
 'activities_activityCalories_std_resid',
 'activities_calories_std_resid',
 'activities_distance_std_resid',
 'activities_elevation_std_resid',
 'activities_floors_std_resid',
 'activities_minutesFairlyActive_std_resid',
 'activities_minutesLightlyActive_std_resid',
 'activities_minutesSedentary_std_resid',
 'activities_minutesVeryActive_std_resid',
 'activities_steps_std_resid']

lifestyle_multiomics_sleep_04-01-2026/lifestyle_and_multi-omics_on_sleep_regression_analysis/lifestyle_and_omics_on_sleep_and_adjusted_10-15-2025/supplement_features_to_sleep_features_log1p_resids_adjusted_10-15-2025.csv

In [20]:
import pandas as pd

# Define feature set names and corresponding file paths
feature_sets = {
    'supplement_features': pd.read_csv('../lifestyle_and_multi-omics_on_sleep_regression_analysis/lifestyle_and_omics_on_sleep_and_adjusted_10-15-2025/supplement_features_to_sleep_features_log1p_resids_adjusted_10-15-2025.csv'),
    'digestion_features': pd.read_csv('../lifestyle_and_multi-omics_on_sleep_regression_analysis/lifestyle_and_omics_on_sleep_and_adjusted_10-15-2025/digestion_features_to_sleep_features_log1p_resids_adjusted_10-15-2025.csv'),
    'CLR_microbe_features': pd.read_csv('../lifestyle_and_multi-omics_on_sleep_regression_analysis/lifestyle_and_omics_on_sleep_and_adjusted_10-15-2025/new_microbe_features_to_sleep_features_log1p_resids_adjusted_10-15-2025.csv'),
    'microbe_presence_absence_features': pd.read_csv('../lifestyle_and_multi-omics_on_sleep_regression_analysis/lifestyle_and_omics_on_sleep_and_adjusted_10-15-2025/new_microbe_binarized_features_to_sleep_features_log1p_resids_adjusted_10-15-2025.csv'),
    'diversity_features': pd.read_csv('../lifestyle_and_multi-omics_on_sleep_regression_analysis/lifestyle_and_omics_on_sleep_and_adjusted_10-15-2025/diversity_features_to_sleep_features_log1p_resids_adjusted_10-15-2025.csv'),
    'clinical_features_log1p': pd.read_csv('../lifestyle_and_multi-omics_on_sleep_regression_analysis/lifestyle_and_omics_on_sleep_and_adjusted_10-15-2025/clinical_features_log1p_to_sleep_features_log1p_resids_adjusted_10-15-2025.csv'),
    'metabolite_features_log': pd.read_csv('../lifestyle_and_multi-omics_on_sleep_regression_analysis/lifestyle_and_omics_on_sleep_and_adjusted_10-15-2025/metabolite_features_log_to_sleep_features_log1p_resids_adjusted_10-15-2025.csv'),
    'proteomics_features': pd.read_csv('../lifestyle_and_multi-omics_on_sleep_regression_analysis/lifestyle_and_omics_on_sleep_and_adjusted_10-15-2025/proteomics_features_to_sleep_features_log1p_resids_adjusted_10-15-2025.csv'),
    'micom_features_log': pd.read_csv('../lifestyle_and_multi-omics_on_sleep_regression_analysis/lifestyle_and_omics_on_sleep_and_adjusted_10-15-2025/micom_features_log_to_sleep_features_log1p_resids_adjusted_10-15-2025.csv')

}

# Extract all unique independent features that were tested in the adjusted models into a dictionary
feature_sets_dict = {
    name: df['independent_feature'].unique().tolist()
    for name, df in feature_sets.items()
}

feature_sets_dict['activity_features_resids'] = features_dict['activity_features_resids']
feature_sets_dict['heartrate_features_resids'] = features_dict['heartrate_features_resids']
feature_sets_dict['control_covariate_features'] = ['age', 'BMI_CALC', 'PC1', 'PC2', 'PC3', 'sex',
                                                   'any_mental_health_self_anytime_before',
                                                   'any_sleep_disorder_self_anytime_before',
                                                   'avg_stress']

# Optional: print keys and counts for inspection
for k, v in feature_sets_dict.items():
    print(f"{k}: {len(v)} features")

supplement_features: 5 features
digestion_features: 16 features
CLR_microbe_features: 177 features
microbe_presence_absence_features: 154 features
diversity_features: 3 features
clinical_features_log1p: 104 features
metabolite_features_log: 1191 features
proteomics_features: 276 features
micom_features_log: 201 features
activity_features_resids: 20 features
heartrate_features_resids: 8 features
control_covariate_features: 9 features


In [23]:
from itertools import chain

# Flatten all the feature lists into one master list
all_features = list(chain.from_iterable(feature_sets_dict.values()))

In [26]:
5+16+177+154+3+104+1191+276+206+20+8+9

2169

In [25]:
len(all_features)

2169

In [27]:
feature_sets_dict['all_features'] = all_features

In [28]:
from itertools import chain

excluded = {"activity_features_resids", "heartrate_features_resids", "all_features"}  # exact key names
kept_keys = [k for k in feature_sets_dict.keys() if k not in excluded]

all_features_minus_activity_hr = list(dict.fromkeys(
    chain.from_iterable(feature_sets_dict[k] for k in kept_keys)
))

In [29]:
kept_keys

['supplement_features',
 'digestion_features',
 'CLR_microbe_features',
 'microbe_presence_absence_features',
 'diversity_features',
 'clinical_features_log1p',
 'metabolite_features_log',
 'proteomics_features',
 'micom_features_log',
 'control_covariate_features']

In [30]:
feature_sets_dict['all_features_minus_activity_heartrate'] = all_features_minus_activity_hr

In [31]:
excluded = {"activity_features_resids", "all_features_minus_activity_heartrate", "all_features"}  # exact key names
kept_keys = [k for k in feature_sets_dict.keys() if k not in excluded]

all_features_minus_activity = list(dict.fromkeys(
    chain.from_iterable(feature_sets_dict[k] for k in kept_keys)
))

In [32]:
feature_sets_dict['all_features_minus_activity'] = all_features_minus_activity

In [33]:
for k, v in feature_sets_dict.items():
    print(f"{k}: {len(v)} features")

supplement_features: 5 features
digestion_features: 16 features
CLR_microbe_features: 177 features
microbe_presence_absence_features: 154 features
diversity_features: 3 features
clinical_features_log1p: 104 features
metabolite_features_log: 1191 features
proteomics_features: 276 features
micom_features_log: 206 features
activity_features_resids: 20 features
heartrate_features_resids: 8 features
control_covariate_features: 9 features
all_features: 2169 features
all_features_minus_activity_heartrate: 2141 features
all_features_minus_activity: 2149 features


In [34]:
feature_sets_dict['control_covariate_features']

['age',
 'BMI_CALC',
 'PC1',
 'PC2',
 'PC3',
 'sex',
 'any_mental_health_self_anytime_before',
 'any_sleep_disorder_self_anytime_before',
 'avg_stress']

# 12-10-2025 Regressing EACH of 18 sleep features on full feature sets. Make a feature set for the control covariates as well

In [37]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RepeatedKFold
from sklearn.metrics import r2_score
import pandas as pd
import numpy as np
import os
    
# -----------------------------
# Config
# -----------------------------
output_dir = 'rf_feature_set_only_repeated_KFold_12-10-2025'
os.makedirs(output_dir, exist_ok=True)
    
base_seed = 42
n_estimators = 500
max_features = 1/3
    
# Repeated K-fold CV settings
n_splits = 4
n_repeats = 10
    
# -----------------------------
# Helpers
# -----------------------------
def impute_train_test(df_train, df_test):
    df_train_imp = df_train.copy()
    df_test_imp  = df_test.copy()

    all_nan_cols = df_train_imp.columns[df_train_imp.isna().all()]
    if len(all_nan_cols) > 0:
        df_train_imp = df_train_imp.drop(columns=all_nan_cols)
        df_test_imp  = df_test_imp.drop(columns=all_nan_cols, errors="ignore")

    impute_stats = {}
    for col in df_train_imp.columns:
        if col.startswith("digestion_"):
            mode_vals = df_train_imp[col].mode(dropna=True)
            val = mode_vals.iloc[0]
        else:
            val = df_train_imp[col].median()

        df_train_imp[col] = df_train_imp[col].fillna(val)
        impute_stats[col] = val

    for col, val in impute_stats.items():
        if col in df_test_imp.columns:
            df_test_imp[col] = df_test_imp[col].fillna(val)

    return df_train_imp, df_test_imp, list(all_nan_cols)

    
def fit_rf(X_tr, y_tr, seed=base_seed):
    rf = RandomForestRegressor(
        n_estimators=n_estimators,
        max_features=max_features,
        n_jobs=-1,
        random_state=seed
    )
    rf.fit(X_tr, y_tr)
    return rf
    
    
def ci95(values):
    lo, hi = np.percentile(values, [2.5, 97.5])
    return float(lo), float(hi)
    
# -----------------------------
# MAIN LOOP: Feature-Set-Only Model
# -----------------------------
for feature_set_name, feature_set in feature_sets_dict.items():
    
    out_summary = os.path.join(output_dir, f'rf_{feature_set_name}_summary.csv')
    out_folds   = os.path.join(output_dir, f'rf_{feature_set_name}_folds.csv')
    
    if os.path.exists(out_summary) and os.path.exists(out_folds):
        print(f"Skipping '{feature_set_name}' (already processed).")
        continue
    
    print(f"\nRunning feature set: {feature_set_name}")
    
    results_rows = []
    fold_rows = []
    
    # Loop over sleep features
    for sleep_feat in sleep_features_vendor_only_resid_rf:
        print(f"  → Sleep feature: {sleep_feat}")
    
        df = sleep_merged_rf.dropna(subset=[sleep_feat]).copy()
    
        X = df[feature_set]
        y = df[sleep_feat].values
    
        rkf = RepeatedKFold(n_splits=n_splits, n_repeats=n_repeats, random_state=base_seed)
    
        r2_all = []
        cv_id = 0

        # For counting and recording dropped all Nan columns in a fold
        dropped_cols_counter = {}
    
        for tr_idx, te_idx in rkf.split(X):
    
            # Split and impute
            X_tr, X_te = X.iloc[tr_idx], X.iloc[te_idx]
            y_tr, y_te = y[tr_idx], y[te_idx]

            X_tr_imp, X_te_imp, dropped_cols = impute_train_test(X_tr, X_te)

            # Record if columns have been dropped due to all Nan in a fold
            for c in dropped_cols:
                dropped_cols_counter[c] = dropped_cols_counter.get(c, 0) + 1


            # Fit
            rf = fit_rf(X_tr_imp, y_tr, seed=base_seed)
            y_pred = rf.predict(X_te_imp)

            r2 = r2_score(y_te, y_pred)
            r2_all.append(r2)

            fold_rows.append({
                'feature_set': feature_set_name,
                'sleep_feature': sleep_feat,
                'cv_id': cv_id,
                'n_test': int(len(te_idx)),
                'r2': float(r2)
            })
            cv_id += 1

        # Summary statistics
        r2_mean = float(np.mean(r2_all))
        r2_std = float(np.std(r2_all))
        r2_lo, r2_hi = ci95(r2_all)

        results_rows.append({
            'sleep_feature': sleep_feat,
            'n_folds': n_splits,
            'n_repeats': n_repeats,
            'r2_mean': r2_mean,
            'r2_std': r2_std,
            'r2_ci_lower': r2_lo,
            'r2_ci_upper': r2_hi,
            'n_features_dropped_any_fold': len(dropped_cols_counter),
            'n_total_drop_events': sum(dropped_cols_counter.values()),
            'dropped_columns_any_fold': ";".join(sorted(dropped_cols_counter.keys()))
        })

    # Save
    pd.DataFrame(results_rows).to_csv(out_summary, index=False)
    pd.DataFrame(fold_rows).to_csv(out_folds, index=False)

print("\nDone.")

Skipping 'supplement_features' (already processed).
Skipping 'digestion_features' (already processed).

Running feature set: CLR_microbe_features
  → Sleep feature: sleep_awakeDuration_log1p_resid_rf_vendor_only_resid
  → Sleep feature: sleep_awakeningsCount_log1p_resid_rf_vendor_only_resid
  → Sleep feature: sleep_efficiency_log1p_resid_rf_vendor_only_resid
  → Sleep feature: sleep_minutesAfterWakeup_log1p_resid_rf_vendor_only_resid
  → Sleep feature: sleep_minutesAsleep_log1p_resid_rf_vendor_only_resid
  → Sleep feature: sleep_restlessDuration_log1p_resid_rf_vendor_only_resid
  → Sleep feature: sleep_timeInBed_log1p_resid_rf_vendor_only_resid
  → Sleep feature: sleep_awakeDuration_std_log1p_resid_rf_vendor_only_resid
  → Sleep feature: sleep_awakeningsCount_std_log1p_resid_rf_vendor_only_resid
  → Sleep feature: sleep_efficiency_std_log1p_resid_rf_vendor_only_resid
  → Sleep feature: sleep_minutesAfterWakeup_std_log1p_resid_rf_vendor_only_resid
  → Sleep feature: sleep_minutesAsleep_

In [47]:
import numpy as np
import pandas as pd
from sklearn.model_selection import RepeatedKFold

def fold_missingness_summary(
    sleep_merged_rf: pd.DataFrame,
    feature_sets_dict: dict,
    sleep_features: list,
    n_splits: int = 4,
    n_repeats: int = 10,
    random_state: int = 42,
    compute_test_missingness: bool = True,
) -> pd.DataFrame:
    """
    Computes average % missingness across CV folds WITHOUT fitting any models.

    Missingness is computed:
      - at the cell level across X_train (and X_test optionally)
      - BEFORE imputation
      - after filtering rows to those with non-missing y (sleep feature), matching your RF pipeline

    Also tracks:
      - columns that are all-NaN in a given training fold (which your imputer would drop)
      - frequency of those drop events

    Returns a tidy summary dataframe suitable for reporting.
    """

    rkf = RepeatedKFold(n_splits=n_splits, n_repeats=n_repeats, random_state=random_state)

    summary_rows = []
    fold_rows = []

    for feature_set_name, feature_cols in feature_sets_dict.items():
        print(f"\nComputing fold missingness for feature set: {feature_set_name}")

        for sleep_feat in sleep_features:
            # Match your pipeline: only keep rows where outcome exists
            df = sleep_merged_rf.dropna(subset=[sleep_feat]).copy()

            # X and y (y only used for row filtering + to define rkf split length)
            X = df[feature_cols]
            y = df[sleep_feat].values

            # Track per-fold missingness and drop events
            train_missing_rates = []
            test_missing_rates = []
            dropped_cols_counter = {}

            cv_id = 0
            for tr_idx, te_idx in rkf.split(X):
                X_tr = X.iloc[tr_idx]
                X_te = X.iloc[te_idx]

                # Columns that are all-NA in THIS training fold (would be dropped in your imputer)
                all_nan_cols = list(X_tr.columns[X_tr.isna().all()])
                for c in all_nan_cols:
                    dropped_cols_counter[c] = dropped_cols_counter.get(c, 0) + 1

                # For missingness rate, compute on the columns that remain after this fold's drop
                X_tr_eff = X_tr.drop(columns=all_nan_cols, errors="ignore")
                X_te_eff = X_te.drop(columns=all_nan_cols, errors="ignore")

                # Cell-level missingness rate (fraction of NA cells)
                tr_miss = float(X_tr_eff.isna().to_numpy().mean()) if X_tr_eff.shape[1] > 0 else np.nan
                train_missing_rates.append(tr_miss)

                if compute_test_missingness:
                    te_miss = float(X_te_eff.isna().to_numpy().mean()) if X_te_eff.shape[1] > 0 else np.nan
                    test_missing_rates.append(te_miss)

                fold_rows.append({
                    "feature_set": feature_set_name,
                    "sleep_feature": sleep_feat,
                    "cv_id": cv_id,
                    "n_train": int(len(tr_idx)),
                    "n_test": int(len(te_idx)),
                    "n_features_total": int(X.shape[1]),
                    "n_features_dropped_this_fold": int(len(all_nan_cols)),
                    "train_missing_rate": tr_miss,
                    "test_missing_rate": te_miss if compute_test_missingness else np.nan,
                })
                cv_id += 1

            # Summaries for this (feature_set, sleep_feature)
            train_mean = float(np.nanmean(train_missing_rates)) if len(train_missing_rates) else np.nan
            train_sd   = float(np.nanstd(train_missing_rates)) if len(train_missing_rates) else np.nan

            if compute_test_missingness:
                test_mean = float(np.nanmean(test_missing_rates)) if len(test_missing_rates) else np.nan
                test_sd   = float(np.nanstd(test_missing_rates)) if len(test_missing_rates) else np.nan
            else:
                test_mean = np.nan
                test_sd = np.nan

            summary_rows.append({
                "feature_set": feature_set_name,
                "sleep_feature": sleep_feat,
                "n_splits": n_splits,
                "n_repeats": n_repeats,
                "n_folds_total": int(n_splits * n_repeats),
                "n_samples_used": int(df.shape[0]),
                "n_features_total": int(X.shape[1]),
                "train_missing_rate_mean": train_mean,
                "train_missing_rate_sd": train_sd,
                "test_missing_rate_mean": test_mean,
                "test_missing_rate_sd": test_sd,
                "n_unique_cols_dropped_any_fold": int(len(dropped_cols_counter)),
                "n_total_drop_events": int(sum(dropped_cols_counter.values())),
                "dropped_columns_any_fold": ";".join(sorted(dropped_cols_counter.keys())),
            })

    summary_df = pd.DataFrame(summary_rows)
    folds_df = pd.DataFrame(fold_rows)

    return summary_df, folds_df


# -----------------------------
# RUN IT
# -----------------------------
missingness_summary_df, missingness_folds_df = fold_missingness_summary(
    sleep_merged_rf=sleep_merged_rf,
    feature_sets_dict=feature_sets_dict,
    sleep_features=sleep_features_vendor_only_resid_rf,
    n_splits=n_splits,
    n_repeats=n_repeats,
    random_state=base_seed,
    compute_test_missingness=True,
)

# Example: overall summary per feature set (averaged across sleep outcomes)
by_feature_set = (
    missingness_summary_df
    .groupby("feature_set", as_index=False)
    .agg(
        train_missing_rate_mean=("train_missing_rate_mean", "mean"),
        train_missing_rate_sd=("train_missing_rate_mean", "std"),
        test_missing_rate_mean=("test_missing_rate_mean", "mean"),
        test_missing_rate_sd=("test_missing_rate_mean", "std"),
        n_unique_cols_dropped_any_fold=("n_unique_cols_dropped_any_fold", "max"),
        n_total_drop_events=("n_total_drop_events", "sum"),
    )
)


Computing fold missingness for feature set: supplement_features

Computing fold missingness for feature set: digestion_features

Computing fold missingness for feature set: CLR_microbe_features

Computing fold missingness for feature set: microbe_presence_absence_features

Computing fold missingness for feature set: diversity_features

Computing fold missingness for feature set: clinical_features_log1p

Computing fold missingness for feature set: metabolite_features_log

Computing fold missingness for feature set: proteomics_features

Computing fold missingness for feature set: micom_features_log

Computing fold missingness for feature set: activity_features_resids

Computing fold missingness for feature set: heartrate_features_resids

Computing fold missingness for feature set: control_covariate_features

Computing fold missingness for feature set: all_features

Computing fold missingness for feature set: all_features_minus_activity_heartrate

Computing fold missingness for feature s

In [54]:
missingness_summary_df.to_csv("rf_feature_set_only_repeated_KFold_12-10-2025/fold_missingness_data_02-03-2026/rf_fold_missingness_summary.csv", index=False)
missingness_folds_df.to_csv("rf_feature_set_only_repeated_KFold_12-10-2025/fold_missingness_data_02-03-2026/rf_fold_missingness_folds.csv", index=False)
by_feature_set.to_csv("rf_feature_set_only_repeated_KFold_12-10-2025/fold_missingness_data_02-03-2026/rf_fold_missingness_by_feature_set.csv", index=False)

# Visualizing results 12-17-2025

In [21]:
import os
from glob import glob
import numpy as np
import pandas as pd
import altair as alt

# -----------------------------
# 1) Build plot_df
# -----------------------------
def make_rf_r2_plot_df(
    output_dir: str,
    summary_glob: str = "rf_*_summary.csv",
    keep_patterns=("minutesAsleep", "timeInBed", "bedtime"),
    require_panel_has_any_good: bool = True,
    keep_only_good_rows: bool = False,
    require_positive_mean: bool = False,
    require_interval_excludes_zero: bool = False,
    faded_opacity: float = 0.25,
    full_opacity: float = 1.0,
) -> pd.DataFrame:
    """
    Loads rf_{feature_set}_summary.csv files and returns a tidy plot_df.

    Interval columns are treated as the central 95% CV-split interval (2.5–97.5th percentile),
    NOT a classical parametric CI.

    Adds:
      - interval_crosses_zero (bool)
      - is_good (bool) based on require_* toggles
      - fade_flag (bool): interval crosses 0 OR mean < 0
      - opacity_val (float)
    """

    files = sorted(glob(os.path.join(output_dir, summary_glob)))
    if len(files) == 0:
        raise FileNotFoundError(f"No summary files matched {summary_glob} in {output_dir}")

    dfs = []
    for f in files:
        base = os.path.basename(f)
        if not (base.startswith("rf_") and base.endswith("_summary.csv")):
            continue
        feature_set_name = base[len("rf_"):-len("_summary.csv")]
        dfi = pd.read_csv(f)
        dfi["feature_set"] = feature_set_name
        dfs.append(dfi)

    if len(dfs) == 0:
        raise FileNotFoundError(f"No valid rf_*_summary.csv files found in {output_dir}")

    df = pd.concat(dfs, ignore_index=True)

    needed = ["sleep_feature", "feature_set", "r2_mean", "r2_ci_lower", "r2_ci_upper"]
    missing = [c for c in needed if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns in summaries: {missing}")

    for c in ["r2_mean", "r2_ci_lower", "r2_ci_upper"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    df = df.dropna(subset=needed).copy()

    # keep only selected sleep features
    if keep_patterns:
        pat = "|".join(map(str, keep_patterns))
        df = df[df["sleep_feature"].astype(str).str.contains(pat, case=False, regex=True, na=False)].copy()

    if df.empty:
        raise ValueError(
            "No rows left after filtering sleep_feature by keep_patterns="
            f"{list(keep_patterns)}"
        )

    # interval crosses 0?
    df["interval_crosses_zero"] = (df["r2_ci_lower"] <= 0) & (df["r2_ci_upper"] >= 0)

    # "good" row? (used for panel inclusion and optional within-panel filtering)
    good = pd.Series(True, index=df.index)
    if require_positive_mean:
        good &= df["r2_mean"] > 0
    if require_interval_excludes_zero:
        good &= ~df["interval_crosses_zero"]
    df["is_good"] = good

    # include only panels with >=1 good row
    if require_panel_has_any_good:
        good_panels = (
            df.groupby("sleep_feature")["is_good"]
              .any()
              .reset_index()
              .query("is_good == True")["sleep_feature"]
              .tolist()
        )
        df = df[df["sleep_feature"].isin(good_panels)].copy()

    if df.empty:
        raise ValueError("No data left after filtering panels. Try require_panel_has_any_good=False.")

    # optionally keep only good rows
    if keep_only_good_rows:
        df = df[df["is_good"]].copy()

    if df.empty:
        raise ValueError(
            "No rows left after keep_only_good_rows=True. "
            "Relax filtering or set keep_only_good_rows=False."
        )

    # fade if (interval crosses 0) OR (mean < 0)
    df["fade_flag"] = df["interval_crosses_zero"] | (df["r2_mean"] < 0)
    df["opacity_val"] = np.where(df["fade_flag"], faded_opacity, full_opacity)

    return df


# -----------------------------
# 2) Plot plot_df
# -----------------------------
def plot_rf_r2_vconcat_from_plot_df(
    plot_df: pd.DataFrame,
    title: str = "Random Forest CV R² by Feature Set (central 95% CV interval)",
    width: int = 520,
    y_step: int = 16,
    section_spacing: int = 10,
    feature_set_col: str = "feature_set",
    sleep_feature_col: str = "sleep_feature",
    mean_col: str = "r2_mean",
    lo_col: str = "r2_ci_lower",
    hi_col: str = "r2_ci_upper",
    opacity_col: str = "opacity_val",
) -> alt.Chart:
    """
    Expects plot_df to already contain:
      - sleep_feature, feature_set, r2_mean, r2_ci_lower, r2_ci_upper
      - opacity_val (float) for fading
    """

    needed = [sleep_feature_col, feature_set_col, mean_col, lo_col, hi_col, opacity_col]
    missing = [c for c in needed if c not in plot_df.columns]
    if missing:
        raise ValueError(f"plot_df missing required columns: {missing}")

    df = plot_df.copy()

    feat_order = list(df[feature_set_col].dropna().unique())
    sleep_order = list(df[sleep_feature_col].dropna().unique())

    zero = alt.Chart(pd.DataFrame({"x": [0.0]})).mark_rule(strokeDash=[4, 4]).encode(x="x:Q")

    charts = []
    for i, sf in enumerate(sleep_order):
        dsub = df[df[sleep_feature_col] == sf].copy()

        # hide x-axis labels for all but bottom panel
        if i < len(sleep_order) - 1:
            x_axis = alt.Axis(labels=False, ticks=False, title=None)
        else:
            x_axis = alt.Axis()

        base = alt.Chart(dsub).encode(
            y=alt.Y(
                f"{feature_set_col}:N",
                sort=feat_order,
                title=None,
                axis=alt.Axis(labelLimit=1200),
            ),
            x=alt.X(
                f"{mean_col}:Q",
                axis=x_axis,
                scale=alt.Scale(zero=False),
                title="R² (Test R² mean; 2.5–97.5th percentile)"
                      if i == len(sleep_order) - 1 else None,
            ),
            color=alt.Color(
                f"{feature_set_col}:N",
                sort=feat_order,
                title="Feature set" if i == 0 else None,
                legend=alt.Legend(
                    orient="right",
                    labelLimit=0,
                    labelFontSize=11,
                    titleFontSize=12,
                    offset=0
                ),
            ),
            opacity=alt.Opacity(f"{opacity_col}:Q", legend=None),
            tooltip=[
                alt.Tooltip(f"{sleep_feature_col}:N", title="Sleep feature"),
                alt.Tooltip(f"{feature_set_col}:N", title="Feature set"),
                alt.Tooltip(f"{mean_col}:Q", title="R² mean", format=".3f"),
                alt.Tooltip(f"{lo_col}:Q", title="Interval lower", format=".3f"),
                alt.Tooltip(f"{hi_col}:Q", title="Interval upper", format=".3f"),
            ],
        )

        rules = base.mark_rule(size=2).encode(
            x=alt.X(f"{lo_col}:Q"),
            x2=f"{hi_col}:Q",
        )

        points = base.mark_point(filled=True, size=70)

        panel = alt.layer(rules, points, zero).properties(
            width=width,
            height={"step": y_step},
            title=alt.TitleParams(
                text=str(sf),
                anchor="middle",
                fontWeight="bold",
                fontSize=13,
                offset=2,
            ),
        )

        charts.append(panel)

    final = alt.vconcat(*charts, spacing=section_spacing).resolve_scale(
        x="shared",
        y="independent",
        color="shared",
        opacity="shared",
    ).properties(
        title=alt.TitleParams(text=title, anchor="start", fontSize=16)
    ).configure_view(
        stroke=None
    ).configure_axis(
        grid=True,
        titleFontWeight="normal"
    )

    return final

In [22]:
# Make the plotting df
plot_df = make_rf_r2_plot_df(
    output_dir="rf_feature_set_only_repeated_KFold_12-10-2025",
    require_panel_has_any_good=True,
    keep_only_good_rows=True,
)

In [23]:
plot_df.sleep_feature.unique()

array(['sleep_minutesAsleep_log1p_resid_rf_vendor_only_resid',
       'sleep_timeInBed_log1p_resid_rf_vendor_only_resid',
       'sleep_minutesAsleep_std_log1p_resid_rf_vendor_only_resid',
       'sleep_timeInBed_std_log1p_resid_rf_vendor_only_resid',
       'bedtime_int_log1p_resid_rf_vendor_only_resid',
       'bedtime_int_std_log1p_resid_rf_vendor_only_resid'], dtype=object)

In [24]:
plot_df.sleep_feature.replace({
    'sleep_minutesAsleep_log1p_resid_rf_vendor_only_resid': 'log1p(minutes_asleep)',
    'sleep_minutesAsleep_std_log1p_resid_rf_vendor_only_resid': 'log1p(minutes_asleep_SD)',
    'sleep_timeInBed_log1p_resid_rf_vendor_only_resid': 'log1p(time_in_bed)',
    'sleep_timeInBed_std_log1p_resid_rf_vendor_only_resid': 'log1p(time_in_bed_SD)',
    'bedtime_int_log1p_resid_rf_vendor_only_resid': 'log1p(bedtime)',
    'bedtime_int_std_log1p_resid_rf_vendor_only_resid': 'log1p(bedtime_SD)'
}, inplace=True)

In [25]:
plot_df.feature_set.unique()

array(['CLR_microbe_features', 'activity_features_resids',
       'all_features_minus_activity_heartrate',
       'all_features_minus_activity', 'all_features',
       'clinical_features_log1p', 'control_covariate_features',
       'digestion_features', 'diversity_features',
       'heartrate_features_resids', 'metabolite_features_log',
       'micom_features_log', 'microbe_presence_absence_features',
       'proteomics_features', 'supplement_features'], dtype=object)

In [26]:
plot_df.feature_set.replace({
    'activity_features_resids': 'Monthly Fitbit Activity Feature Set',
    'all_features_minus_activity': 'All Feature Sets Except Activity',
    'all_features_minus_activity_heartrate': 'All Feature Sets Except Activity and Heart Rate',
    'all_features': 'All Feature Sets Combined'
}, inplace=True)

In [27]:
plot_df[plot_df.feature_set.str.contains('Activity|All')].feature_set.unique()

array(['Monthly Fitbit Activity Feature Set',
       'All Feature Sets Except Activity and Heart Rate',
       'All Feature Sets Except Activity', 'All Feature Sets Combined'],
      dtype=object)

In [28]:
plot_df

,sleep_feature,n_folds,n_repeats,r2_mean,r2_std,r2_ci_lower,r2_ci_upper,n_features_dropped_any_fold,n_total_drop_events,dropped_columns_any_fold,feature_set,interval_crosses_zero,is_good,fade_flag,opacity_val
4,log1p(minutes_asleep),4,10,-0.009967,0.042493,-0.081627,0.050456,0.0,0.0,NaN,CLR_microbe_features,True,True,True,0.25
7,log1p(time_in_bed),4,10,0.001658,0.042372,-0.090838,0.052321,0.0,0.0,NaN,CLR_microbe_features,True,True,True,0.25
12,log1p(minutes_asleep_SD),4,10,0.000952,0.019667,-0.035904,0.033277,0.0,0.0,NaN,CLR_microbe_features,True,True,True,0.25
15,log1p(time_in_bed_SD),4,10,-0.026467,0.037082,-0.082046,0.036694,0.0,0.0,NaN,CLR_microbe_features,True,True,True,0.25
16,log1p(bedtime),4,10,0.013266,0.014639,-0.017703,0.032719,0.0,0.0,NaN,CLR_microbe_features,True,True,True,0.25
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
259,log1p(time_in_bed),4,10,-0.083720,0.052770,-0.157859,-0.019730,NaN,NaN,NaN,supplement_features,False,True,True,0.25
264,log1p(minutes_asleep_SD),4,10,-0.059599,0.041361,-0.147806,0.012666,NaN,NaN,NaN,supplement_features,True,True,True,0.25
267,log1p(time_in_bed_SD),4,10,-0.086684,0.047787,-0.180008,-0.010132,NaN,NaN,NaN,supplement_features,False,True,True,0.25
268,log1p(bedtime),4,10,-0.022510,0.021275,-0.071405,0.016648,NaN,NaN,NaN,supplement_features,True,True,True,0.25


In [32]:
# Plot the plot_df all results to see which feature sets are non-zero at least once
chart = plot_rf_r2_vconcat_from_plot_df(
    plot_df,
    width=150,
    y_step=14,
    section_spacing=8,
    title="Random Forest CV R² by Feature Set (central 95% CV interval)",
)

chart

alt.VConcatChart(...)

In [30]:
# Plot only the feature sets that are non-zero at least once, aka any feature set containing activity and the all feature set minus activity
chart = plot_rf_r2_vconcat_from_plot_df(
    plot_df[plot_df.feature_set.str.contains('Activity|All')],
    width=150,
    y_step=14,
    section_spacing=8,
    title="Random Forest CV R² by Feature Set (central 95% CV interval)",
)

chart

alt.VConcatChart(...)